## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
# from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
# load_dotenv(override=True)

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [ ]:
BASE_URL = "https://models.github.ai/inference"
MODEL_GPT_4o_MINI = "gpt-4o-mini"
MODEL_NAME = MODEL_GPT_4o_MINI
API_KEY = "github_pat_******"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    set_default_openai_client,
    set_default_openai_api
)

github_client = AsyncOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

set_default_openai_client(client=github_client, use_for_tracing=False)
set_default_openai_api("chat_completions")

In [4]:
# Make an agent with name, instructions, model
agent = Agent(
  name="Jokester", 
  instructions="You are a joke teller",
  model=MODEL_NAME
  )

In [5]:
# Run the joke with Runner.run(agent, prompt)
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [6]:
# Here is the final output
print(result.final_output)

Why did the autonomous AI agent break up with its partner?

Because it couldn’t handle the emotional bandwidth!


In [7]:
# Here is the detail of the LLM calls
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent break up with its partner?\n\nBecause it couldn’t handle the emotional bandwidth!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gpt-4o-mini',
   'response_id': 'chatcmpl-DrEPwGujtGx69IzR4vmZK19C22po0'}}]

## Adding Observability with a trace

In [8]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent break up with its human partner?

Because it just couldn't handle the emotional bandwidth!


## Now go and look at the trace

https://platform.openai.com/traces

In [9]:
# Streaming
result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure! Here are five jokes about AI agents for you:

1. Why did the AI agent break up with its partner?
   Because it couldn’t handle the “emotional algorithms”!

2. How do AI agents stay motivated?
   They always find a way to "byte" off more than they can chew!

3. Why did the AI agent go to therapy?
   It had too many unresolved neural network issues!

4. What do you call an AI that tells bad jokes?
   A pun-derful machine learning model!

5. Why did the human get mad at their AI assistant?
   Because it kept trying to "intelligently" suggest their coffee order was too basic!

I hope these brought a smile to your face!

## Part 2: Adding a tool

In [ ]:
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
# if pushover_user:
#     if pushover_user.startswith("u"):
#         print("Pushover user found and looks good")
#     else:
#         print("Pushover user found but doesn't start with u")
# else:
#     print("Pushover user not found")

# if pushover_token:
#     if pushover_token.startswith("a"):
#         print("Pushover token found and looks good")
#     else:
#         print("Pushover token found but doesn't start with a")
# else:
#     print("Pushover token not found")

In [ ]:
# Remember this?
# def push(message):
#     print(f"Push: {message}")
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     requests.post(pushover_url, data=payload)

def push(message):
    print(f"🔥 [PUSH EMULATION]: {message}")


In [13]:
push("HEY!!")

🔥 [PUSH EMULATION]: HEY!!


In [ ]:
push

In [17]:
# Now this:

# @function_tool
# def push_tool(message: str) -> str:
#     """ Send the given message to the user as a push notification """
#     payload = {"user": pushover_user, "token": pushover_token, "message": message}
#     result = requests.post(pushover_url, data=payload).status_code
#     return f"Push sent with API status code {result}"

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    print(f"\n🔥 [PUSH EMULATION]: {message}\n")
    return "Push sent with API status code 200"

In [18]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7881ec6fb5c0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [19]:
push_tool.description

'Send the given message to the user as a push notification'

In [ ]:
notifier = Agent(name="Notifier", model=MODEL_NAME, instructions="You notify the user upon request", tools=[push_tool])

In [21]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)



🔥 [PUSH EMULATION]: Your pizza has arrived!

The user has been notified that their pizza has arrived!


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [22]:
agent = Agent(name="Assistant", model=MODEL_NAME)

In [23]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! How can I assist you today?


In [24]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. So, I don't know your name. If you’d like to share it or if you have any other questions, feel free to let me know!


## Memory approach 1 - just manually pass in the list of dicts

In [25]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! How can I assist you today?


In [26]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gpt-4o-mini',
   'response_id': 'chatcmpl-DrEwxku1N3LBYB4vWeuJGaLBxk7Me'}}]

In [27]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hi Ed! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gpt-4o-mini',
   'response_id': 'chatcmpl-DrEwxku1N3LBYB4vWeuJGaLBxk7Me'}},
 {'role': 'user', 'content': "What's my name?"}]

In [28]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed. How can I help you today, Ed?


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [29]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [30]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed! It's nice to meet you. How can I assist you today?


In [31]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed. How can I help you today?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..
